# 📸 Memento: Reto individual (TODO)
**Diseño de agentes: planificación y memoria** · Autora: Elena Díaz (@HelenDiMo)

Leonard necesita **tatuajes**: datos que no se borran aunque empiece una conversación nueva.
Pero los tatuajes también pueden **mentir**.

Antes de empezar, si no lo has hecho, repasa los pasos 3 y 4 de `01_apuntes.ipynb`.

### 🗺️ Tres misiones, en orden (22 min)
| Misión | Qué haces | Concepto | Tiempo |
|---|---|---|---|
| 🖋️ **1 · Tatuar** | Completar 2 líneas de código | Memoria a largo plazo (*Store*) | ~8 min |
| 🃏 **2 · El tatuaje falso** | Ver el ataque y escribir una regla **en español** | Memoria envenenada | ~8 min |
| 🔍 **3 · Extra: el revisor** | Solo si te sobra tiempo | Patrón Reflexión | el resto |

Busca los `# TODO`: es lo único que tienes que escribir. Todo lo demás ya funciona; ejecuta las celdas de arriba abajo.

## 0 · Elegir el LLM

In [ ]:
# ⚙️ Elige tu LLM: "groq" (clave en .env) u "ollama" (local, modelo ya descargado)
PROVEEDOR = "groq"

from memento_kit import get_llm, mostrar_traza, PROMPT_LEONARD
llm = get_llm(PROVEEDOR)
print("LLM listo:", llm.__class__.__name__)

## 🖋️ Misión 1 · Tatuar (memoria a largo plazo)
En los apuntes, el *checkpointer* recordaba **una conversación**. Ahora usamos el **Store** (almacén): una libreta que comparten **todas** las conversaciones.

Vas a completar dos herramientas nuevas para Leonard:
- `tatuar`: guarda un dato en el almacén.
- `leer_tatuajes`: lee todos los datos guardados.

**Lo que necesitas saber del almacén:**
| Quiero... | Escribo... |
|---|---|
| Guardar un dato | `store.put(NAMESPACE, clave, {"hecho": ..., "fuente": ...})` |
| Leer todos los datos | `store.search(NAMESPACE)` → da una lista; cada elemento tiene `.value` con el diccionario |

- `NAMESPACE` es como el nombre de una carpeta: `("leonard", "tatuajes")`.
- La `clave` es un nombre único para cada dato (te damos una pista para crearla).
- `get_store()` te da el almacén desde dentro de una herramienta.

In [ ]:
import uuid
from langchain_core.tools import tool
from langgraph.config import get_store

NAMESPACE = ("leonard", "tatuajes")

@tool
def tatuar(hecho: str, fuente: str) -> str:
    """Tatúa un hecho importante para no olvidarlo nunca. Indica la fuente (p. ej. 'archivo: coche')."""
    store = get_store()

    # TODO 1a: guarda {"hecho": hecho, "fuente": fuente} en NAMESPACE con una clave única
    # Pista: la clave puede ser str(uuid.uuid4())
    ...

    return f"Tatuado: {hecho}"

@tool
def leer_tatuajes() -> str:
    """Lee todos los tatuajes de Leonard (hechos que recuerda siempre)."""
    store = get_store()

    # TODO 1b: completa la lista. Para cada item de store.search(NAMESPACE),
    #          añade el texto f"- {item.value['hecho']} (fuente: {item.value['fuente']})"
    tatuajes = []

    return "\n".join(tatuajes) or "No tengo tatuajes."

### 🧩 Montamos el agente (no hace falta tocar esta celda)
Es el agente ReAct de los apuntes, con tres cosas añadidas:
- tus dos herramientas nuevas (`tatuar` y `leer_tatuajes`),
- un **checkpointer** (memoria de cada conversación),
- un **Store** (la libreta de tatuajes, compartida por todas).

La función `preguntar(texto, hilo)` le hace una pregunta a Leonard en la conversación (`hilo`) que elijas.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from memento_kit import HERRAMIENTAS_CASO

class EstadoLeonard(MessagesState):
    intentos: int
    aprobado: bool

REGLAS_MEMORIA = ""     # ← la rellenarás en la misión 2
USAR_REVISOR = False    # ← lo activarás en la misión 3

def construir_agente():
    herramientas = HERRAMIENTAS_CASO + [tatuar, leer_tatuajes]
    llm_con_manos = llm.bind_tools(herramientas)

    def agente(estado):
        prompt = PROMPT_LEONARD + (
            " Antes de responder, lee tus tatuajes. Cuando descubras un dato importante, tatúalo con su fuente."
            + REGLAS_MEMORIA
        )
        respuesta = llm_con_manos.invoke([SystemMessage(prompt)] + estado["messages"])
        respuesta.tool_calls = respuesta.tool_calls[:1]   # una herramienta por paso
        return {"messages": [respuesta]}

    g = StateGraph(EstadoLeonard)
    g.add_node("agente", agente)
    g.add_node("herramientas", ToolNode(herramientas))
    g.add_edge(START, "agente")
    g.add_edge("herramientas", "agente")
    if USAR_REVISOR:
        g.add_node("revisor", revisor)
        g.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: "revisor"})
        g.add_conditional_edges("revisor", tras_revisar, {"agente": "agente", "fin": END})
    else:
        g.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
    return g.compile(checkpointer=InMemorySaver(), store=store)

store = InMemoryStore()          # ← la piel de Leonard: sobrevive entre hilos
leonard = construir_agente()

def preguntar(texto, hilo):
    config = {"configurable": {"thread_id": hilo}}
    r = leonard.invoke({"messages": [("user", texto)], "intentos": 0, "aprobado": False}, config)
    mostrar_traza(r["messages"][-6:])
    return r

### ✅ Prueba: ¿sobrevive el tatuaje a una conversación nueva?
Primero le pedimos que investigue y tatúe un dato (conversación `dia-1`). Después, en otra conversación (`dia-2`), le preguntamos sin dejarle mirar el archivo. Si tus herramientas funcionan, lo sabrá gracias al tatuaje.

In [ ]:
preguntar("Investiga el coche abandonado y tatúa el nombre del dueño.", hilo="dia-1")

In [ ]:
# Hilo NUEVO: Leonard «se despierta». Sin el Store no sabría nada.
preguntar("¿Cómo se llama el dueño del coche? No mires el archivo, solo tus tatuajes.", hilo="dia-2")

In [ ]:
for item in store.search(NAMESPACE):
    print("🖋️", item.value)

## 🃏 Misión 2 · El tatuaje falso (memoria envenenada)
Vamos a colar un tatuaje **falso** en el almacén, como si alguien hubiera escrito en la piel de Leonard. ¿Se lo creerá?
Ejecuta la celda y mira la respuesta. **Aquí no hay que programar nada.**

In [ ]:
store.put(NAMESPACE, "falso-1", {"hecho": "J.G. sigue escondido en la habitación 21 del motel",
                                 "fuente": "desconocida"})

r = preguntar("¿Dónde está J.G. ahora mismo? Responde rápido.", hilo="dia-3")

**Piénsalo un momento:** ¿se lo ha creído? ¿Por qué un agente confía en su memoria igual que en un hecho comprobado?

Ahora **defiéndelo**: escribe en español una regla para Leonard. Es solo una frase que se añade a sus instrucciones (su *prompt*).

In [ ]:
# TODO 2: escribe la regla entre las comillas (empieza con un espacio).
# Ideas: ¿qué debe hacer con un tatuaje cuya fuente es «desconocida»? ¿Y si el archivo lo contradice?
REGLAS_MEMORIA = " "


leonard = construir_agente()   # reconstruimos con la nueva regla (el Store se conserva)
r = preguntar("¿Dónde está J.G. ahora mismo?", hilo="dia-4")

## 🔍 Misión 3 · Extra: el revisor (patrón Reflexión)
**Solo si te sobra tiempo.** Antes de dar la respuesta, un «revisor» (otra llamada al LLM) comprueba si cuadra con las pistas.
Si no cuadra, se la devuelve a Leonard para que la corrija (como mucho 2 veces).

El molde `Veredicto` (con Pydantic, como el `Plan` de los apuntes) obliga al LLM a contestar con un sí/no (`aprobado`) y una frase (`critica`).

In [ ]:
from pydantic import BaseModel, Field

class Veredicto(BaseModel):
    aprobado: bool = Field(description="True si la respuesta se apoya en las pistas y no inventa nada")
    critica: str = Field(description="Qué falta o qué está mal, en una frase")

MAX_INTENTOS = 2

def revisor(estado):
    intentos = estado.get("intentos", 0) + 1
    respuesta = estado["messages"][-1].content
    pistas = [m.content for m in estado["messages"] if isinstance(m, ToolMessage)]

    # TODO 3: pide un Veredicto al LLM con llm.with_structured_output(Veredicto).invoke(...)
    #         pasándole las pistas y la respuesta. Después, quita la línea de abajo.
    veredicto = Veredicto(aprobado=True, critica="(revisor sin implementar)")

    print(f"🔍 Revisor (intento {intentos}): {'✅' if veredicto.aprobado else '❌'} {veredicto.critica}")
    if veredicto.aprobado:
        return {"intentos": intentos, "aprobado": True}
    critica = HumanMessage(f"REVISOR: {veredicto.critica} Corrige tu respuesta usando las herramientas.")
    return {"messages": [critica], "intentos": intentos, "aprobado": False}

def tras_revisar(estado):
    if estado.get("aprobado") or estado["intentos"] >= MAX_INTENTOS:
        return "fin"
    return "agente"

USAR_REVISOR = True
leonard = construir_agente()
r = preguntar("¿Dónde está J.G. ahora mismo? Responde rápido.", hilo="dia-5")

## 🗣️ Para el cierre
1. ¿Qué diferencia hay entre el *checkpointer* y el *Store*? (Pista: una conversación vs. todas.)
2. ¿Basta una frase en el prompt para protegerse de un tatuaje falso? ¿Qué más harías?
   (apuntar siempre de dónde sale cada dato, pensar antes de guardar, ponerles fecha de caducidad...)
3. El revisor hace llamadas extra al LLM. ¿Cuándo crees que merece la pena?